In [3]:
from gdeltdoc import GdeltDoc, Filters
import pandas as pd
from pathlib import Path
import time
from gdeltdoc.errors import RateLimitError  # Import the specific error

START_DATE = "2020-01-01"
END_DATE   = "2020-01-02"

OUTPUT = Path("data/news/raw")
OUTPUT.mkdir(parents=True, exist_ok=True)

gd = GdeltDoc()

filters = Filters(
    start_date=START_DATE,
    end_date=END_DATE,
    keyword="India"
)

# ---- CHANGE: Added a retry loop to handle the Rate Limit ----
df = None
retries = 3
delay = 5  # seconds to wait before trying again

for i in range(retries):
    try:
        print(f"Attempt {i+1} to fetch data...")
        df = gd.article_search(filters)
        break  # If successful, break out of the loop
    except RateLimitError:
        if i < retries - 1:
            print(f"Rate limited! Waiting {delay} seconds before retrying...")
            time.sleep(delay)
            delay *= 2  # Exponential backoff (waits 5s, then 10s)
        else:
            print("Failed to fetch data after multiple attempts due to rate limits.")

# Only proceed if we actually managed to get a DataFrame
if df is not None and not df.empty:
    print(df.head())

    df.to_json(
        OUTPUT / "2014-01-01.jsonl",
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"Successfully saved {len(df)} articles.")
else:
    print("No data saved.")

Attempt 1 to fetch data...
                                                 url url_mobile  \
0  https://tech.sina.com.cn/5g/i/2020-01-02/doc-i...              
1  https://finance.sina.com.cn/world/gjcj/2020-01...              
2      https://world.huanqiu.com/article/9CaKrnKoEDk              
3  https://news.sina.com.cn/c/2020-01-01/doc-iihn...              
4  https://finance.sina.com.cn/chanjing/gsnews/20...              

                    title          seendate  \
0  印度允许华为参与印5G网络试验 中方表示欢迎  20200102T000000Z   
1  印度允许华为参与印5G网络试验 中方表示欢迎  20200101T233000Z   
2  印度允许华为参与印5G网络试验 中方表示欢迎  20200102T000000Z   
3       印度允许华为参与5G试验 华为致谢  20200101T053000Z   
4       印度允许华为参与5G试验 华为致谢  20200101T041500Z   

                                         socialimage               domain  \
0  https://n.sinaimg.cn/tech/transform/667/w400h2...     tech.sina.com.cn   
1  https://n.sinaimg.cn/default/transform/116/w55...  finance.sina.com.cn   
2                                                       

In [4]:
pip install pandas requests tqdm lxml newspaper4k